# tiny-log-parser v3 — try it, then check it

A Qwen3-4B fine-tune that normalises a log line into a canonical seven-field
JSON record, emitting `null` for fields the line does not carry rather than
inventing them.

**Part 1 — try it on your own logs.** Paste your lines, watch v3 parse them,
then put `gemini-3.1-pro-preview` beside it on the identical prompt. v3 is free
on this T4; Gemini is optional and costs ~$0.011/line.

**Part 2 — the measured result.** 500 held-out lines from the ten systems v3
trained on. The notebook proves the train/eval split is clean **before** it
prints any accuracy number: training drew from Loghub-2.0, evaluation from
Loghub-1.0, and `measure_contamination.py` confirms **0/500** overlap at both
line and template level.

**The result, so you know what you are re-running.** On 60 clean held-out
lines, adjudicated against `ADJUDICATION.md`:

| arm | correct | accuracy |
|---|---|---|
| `rule_parser.py` — 250 lines of regex | 60/60 | **100.0%** |
| gemini-3.1-pro-preview | 58/60 | **96.7%** |
| v3 — 4B fine-tune | 54/60 | **90.0%** |

**v3 does not beat Gemini** (McNemar p = 0.125), and a hand-written parser beats
both. What v3 buys is the same task at **zero marginal cost** on a free T4, and
all six of its errors trace to three named training-data gaps —
`real-eval/RESULTS_HELDOUT.md` has the breakdown.

**Runtime → Change runtime type → T4 GPU.** Six cells.

Repo: https://github.com/arshirazi97/tiny-log-parser


In [ ]:
# 1. setup (~3 min) -- restarts once, then re-run this same cell
import os
os.chdir('/content')
!rm -rf tiny-log-parser
!git clone -q https://github.com/arshirazi97/tiny-log-parser.git
os.chdir('/content/tiny-log-parser')
!pip install -q "transformers==4.51.3" "peft==0.20.0" accelerate bitsandbytes openai
!pip uninstall -q -y torchao      # peft 0.20 raises on torchao < 0.16; Colab ships 0.10

import importlib.util, transformers, peft
stale = ((transformers.__version__, peft.__version__) != ("4.51.3", "0.20.0")
         or importlib.util.find_spec("torchao") is not None)
if stale:
    print("restarting to pick up the new versions...")
    os.kill(os.getpid(), 9)

---

## Bring your own logs

**Run the next cell and it does everything**: asks for your log lines, parses
them with the deterministic parser and with v3, then asks for an OpenRouter key
and puts `gemini-3.1-pro-preview` beside them. Press Enter at the key prompt to
skip Gemini and see v3 alone, for free.

Paste lines from your own Hadoop, Spark, Linux, HDFS, Apache, Zookeeper,
HealthApp, OpenSSH, OpenStack or Proxifier deployment — the ten systems v3 was
trained on, and the case it should handle well. **Ten or more** makes a better
demo than three.

Gemini bills about **$0.011/line** (reasoning tokens count as output), so ten
lines is roughly $0.11. v3 runs free on this T4.

No gold labels, so nothing here is an accuracy number — read the output against
`schema_v2.SPEC_EVAL` and judge it yourself. Two arms agreeing can both be wrong.


In [ ]:
# 2. YOUR OWN LOGS -- paste, parse with v3, then compare against Gemini
#
# Run this cell. It asks for your log lines, then for an OpenRouter key.
# Everything it needs was cloned from GitHub in cell 1.
import getpass, os, json
os.chdir('/content/tiny-log-parser')

MIN_LINES = 10
print(f'Paste your log lines below, one per line ({MIN_LINES}+ recommended).')
print('You can paste them all at once.')
print('Press Enter on an EMPTY line when you are done.\n')

lines = []
while True:
    try:
        l = input()
    except EOFError:
        break
    if not l.strip():
        break
    lines.append(l.rstrip())

assert lines, 'No lines captured -- re-run the cell and paste your logs.'
print(f'\n>>> {len(lines)} lines captured')
if len(lines) < MIN_LINES:
    print(f'    (fewer than {MIN_LINES}; fine for a look, too few to conclude anything)')

open('mine.log', 'w').write('\n'.join(lines) + '\n')

print('\n>>> building the corpus ...')
!python3 real-eval/messy_corpus.py mine.log -o real-eval/corpus_mine.jsonl

print('\n>>> the deterministic parser (free, instant) ...')
!python3 real-eval/predict.py --arm rules --corpus real-eval/corpus_mine.jsonl \
    --out real-eval/preds_mine_rules.jsonl

print('\n>>> OUR MODEL, v3 (free on this T4) ...')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!python3 real-eval/predict.py --arm model --adapter arshirazi/tiny-log-parser-v3 \
    --corpus real-eval/corpus_mine.jsonl --batch 8 \
    --out real-eval/preds_mine_v3.jsonl

# --shots 0: both arms get SPEC_EVAL and nothing else. With few-shot examples
# you would be comparing two prompts rather than two models.
cost = 0.011 * len(lines)
print(f'\n>>> Gemini next. Costs about ${cost:.2f} for {len(lines)} lines.')
key = getpass.getpass('OpenRouter API key (press Enter to skip): ').strip()
arms = ('rules=real-eval/preds_mine_rules.jsonl '
        'v3=real-eval/preds_mine_v3.jsonl')
if key:
    os.environ['OPENROUTER_API_KEY'] = key
    !python3 real-eval/predict.py --arm gemini \
        --gemini-model google/gemini-3.1-pro-preview --shots 0 \
        --corpus real-eval/corpus_mine.jsonl \
        --out real-eval/preds_mine_gemini.jsonl
    arms += ' gemini=real-eval/preds_mine_gemini.jsonl'
else:
    print('skipped -- comparing v3 against the parser only.')

print('\n' + '=' * 92)
print('COMPARISON')
print('=' * 92)
!python3 real-eval/compare_arms.py --corpus real-eval/corpus_mine.jsonl {arms}


### What to expect, honestly

- **This is inspection, not measurement.** No gold labels, so nothing here is
  an accuracy number. Agreement between the two arms is not proof either —
  both can be wrong on the same line.
- **Your format may not be Loghub's format.** v3 learned *Loghub's* Hadoop, not
  every Hadoop. A different appender layout is a different distribution, and
  the ten system names are not a guarantee.
- **Off these ten formats it degrades, and we know where.** JSON, logfmt and
  Apache combined access logs appear **zero** times in v3's 10,728 training
  examples. So do `<PRI>` syslog prefixes, epoch timestamps and `tid=<32 hex>`.
  Feed it those and it will show.
- **`latency_ms`, `trace_id` and `status_code` are the fragile three.** Non-null
  in 2.97%, 3.17% and 1.79% of training, so v3's learned prior is to abstain.
  Right on this distribution, wrong off it. Every non-null latency it ever saw
  used the notation `time: <n>` — `took=`, `rt=`, `elapsed=` and `duration_us`
  are untrained territory.

The parser (`rules`) is the useful contrast: it scores 100% on these ten
formats and fails to parse 19 of 20 lines in `messy.log`. It is a hand-written
ceiling that does not move. That is the thing the fine-tune is trying to buy
its way out of.

---

# Part 2 — the held-out evaluation

The section above is a demonstration you judge by eye. This one is the measured
result: 500 lines from the ten systems v3 trained on, with **0/500 template
overlap** verified before any accuracy number is printed.


In [ ]:
# 3. prove the split is clean BEFORE looking at any accuracy number
import os
os.chdir('/content/tiny-log-parser')

# the 50-line corpus is a stratified subset of the 500 -- 5 lines from each of
# the ten systems. For the full run drop the '50' from both names.
CORPUS = 'real-eval/corpus_heldout50.jsonl'
RULES  = 'real-eval/preds_heldout50_rules.jsonl'
V3OUT  = 'real-eval/preds_heldout50_v3.jsonl'

!python3 v3/measure_contamination.py --eval {CORPUS}


Expect `0/50` in both the `exact` and `same tmpl` columns. That is the reply
to "you benchmarked on your training data" — established before the result is
known, not after.


In [ ]:
# 4. v3 on the held-out lines (~2 min, mostly model load)
#
# SPEC_EVAL (~815 tokens) rides in every prompt, so activation memory is
# batch * ~900 tokens no matter how short the log lines are. A T4 OOMs
# around batch 32. predict.py now halves the batch and retries on OOM, so
# this completes either way -- 8 just avoids the wasted first attempt.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!python3 real-eval/predict.py --arm model --adapter arshirazi/tiny-log-parser-v3 \
    --corpus {CORPUS} --batch 8 --out {V3OUT}

Watch the `unparseable` count. On in-distribution formats it should be 0 —
v3 emitted 1 unparseable in 262 lines on P1.

In [ ]:
# 5. v3 against the rules reference -- only where they disagree
!python3 real-eval/compare_arms.py --corpus {CORPUS} \
    --only-disagreements rules={RULES} v3={V3OUT}

## Reading the result

**There are no gold labels here and none are invented.** The method is
adjudicated disagreement: diff the arms, hand-check only where they differ.

That is licensed by one fact and does not generalise: `rule_parser.py` scored
**60/60** on the clean held-out lines, adjudicated against `ADJUDICATION.md`.
It is a reference *on this distribution* and nowhere else — on `messy.log` it
fails to parse 19 of 20 lines.

- **Agreement is a proxy, not a measurement.** Both arms can be wrong on the
  same line.
- **Every disagreement needs a human.** Judge against `schema_v2.SPEC_EVAL` and
  `ADJUDICATION.md`; expect a handful, and expect some to be the parser's fault.
- **Watch `status_code`, `trace_id`, `latency_ms`.** Non-null in 1.79%, 3.17%
  and 2.97% of training, so v3's learned prior is to abstain — right on this
  distribution, wrong off it. A disagreement here is the interesting one.

**Calibration.** On 60 clean held-out lines (0/60 exact and template overlap),
adjudicated: `rule_parser.py` **100.0%**, gemini-3.1-pro **96.7%**, v3
**90.0%**. v3 does not beat Gemini; McNemar p = 0.125, and all six of its errors
trace to three named training-data gaps. See `real-eval/RESULTS_HELDOUT.md`.

An earlier 99.0% figure from the P1 corpus is **superseded** — that corpus is
21.9% template-contaminated, found by the fixed contamination guard and
discarded. `real-eval/CONTAMINATION.md` has the audit.

## Adding the frontier arms to the 500

```python
import getpass, os
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ').strip()

# gemini alone on 500 lines is ~$5.50 at the measured $0.011/line
!python3 real-eval/predict.py --arm gemini \
    --gemini-model google/gemini-3.1-pro-preview --shots 0 \
    --corpus {CORPUS} --out real-eval/preds_heldout50_gemini.jsonl
```

Then adjudicate only the lines where the arms disagree, and score.
